# Libraries

In [23]:
import joblib
import numpy as np
import pandas as pd
from UQpy.distributions import Uniform, Normal, JointIndependent #, Lognormal

# Run PCE

### Realizations of $R$ and $S$

In [24]:
n_samples = 1000
n_latent_samples = 5000
time = [0., 10., 20., 30., 40., 50., 60., 70., 80., 90., 100.]
r   = Normal(loc = 5., scale=0.8)
s   = Normal(loc = 2., scale=0.6)
joint = JointIndependent(marginals=[r, s])
x = joint.rvs(n_samples)

In [25]:
# Load the PCE metamodels
times = [0., 10., 20., 30., 40., 50., 60., 70., 80., 90., 100.]
pces = []
for i in times:
    pce_i = joblib.load(f'pce_metamodel_{int(i)}.pkl')
    pces.append(pce_i)

In [26]:
lambdas = []
for i in range(len(times)):
    lambdas.append(pces[i].predict(x))

In [ ]:
dados_completos = []
for i in range(len(times)):
    t_atual = times[i]  # Pega o inteiro do tempo (ex: 0, 5, 10...)
    
    # 1. Descobre quantas amostras (linhas) tem no x
    n_linhas = x.shape[0] 
    
    # 2. Cria uma coluna vertical repetindo esse tempo
    coluna_tempo = np.full((n_linhas, 1), t_atual)
    
    # 3. Concatena: [ X | Lambda_i | Tempo ]
    bloco = np.concatenate((x, lambdas[i], coluna_tempo), axis=1)
    dados_completos.append(bloco)
matriz_final = np.vstack(dados_completos)
nomes_colunas = ['R', 'S', 'lambda1', 'lambda2', 'lambda3', 'lambda4', 'Time']
df = pd.DataFrame(matriz_final, columns=nomes_colunas)
df

,R,S,lambda1,lambda2,lambda3,lambda4,Time
0,4.868881,2.102861,2.777355,6.080100,0.102221,0.158011,0.0
1,4.929642,1.801160,3.137853,6.683432,0.108522,0.151883,0.0
2,5.202769,1.846639,3.365597,6.445516,0.109630,0.150767,0.0
3,4.053031,0.678442,3.376250,11.095017,0.131990,0.127112,0.0
4,5.079017,2.179674,2.911013,5.855615,0.102461,0.157744,0.0
...,...,...,...,...,...,...,...
10995,4.633271,2.212716,-0.809798,6.683070,0.080934,0.186605,100.0
10996,4.276812,2.027925,-0.733089,7.241846,0.081289,0.186766,100.0
10997,5.756485,1.733627,0.003786,8.467466,0.084428,0.181674,100.0
10998,4.437667,2.555249,-1.209045,5.960441,0.080000,0.186295,100.0
